In [ ]:
import sqlite3
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cbook import boxplot_stats
from matplotlib.figure import Figure

from churn_analysis.config import DB_PATH, DB_TABLE_NAME

In [ ]:
# SET DATA SCHEMA FOR PANDAS SQL READER
SCHEMA_MAP = {
    "customer_id": "string",
    "gender": "string",
    "senior_citizen": "boolean",
    "partner": "boolean",
    "dependents": "boolean",
    "tenure": "Int64",
    "phone_service": "boolean",
    "multiple_lines": "string",
    "internet_services": "string",
    "online_security": "boolean",
    "online_backup": "boolean",
    "device_protection": "boolean",
    "tech_support": "boolean",
    "streaming_tv": "boolean",
    "streaming_movies": "boolean",
    "contract": "string",
    "paper_less_billing": "boolean",
    "payment_method": "string",
    "monthly_charges": "Float64",
    "total_charges": "Float64",
    "churn": "boolean",
}


def execute_query(query: str) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        result = pd.read_sql_query(sql=query, con=conn, dtype=SCHEMA_MAP)
    return result

In [ ]:
# MISSING VALUES WAS HANDLED IN TRANSFORM STEP WITHIN ETL PHASE
# THERE ARE NO DUPLICATED RECORDS AS WELL
df = execute_query(f"SELECT * FROM {DB_TABLE_NAME}")
NUMB_ROWS = df.index.size
print(f"Duplicated records: {df.duplicated().sum()}")
df = df.drop(columns="customer_id")

In [ ]:
# CHURN COUNTS
df["churn"].value_counts()

In [ ]:
# PLOT BOXPLOT GRAPH FOR NUMERIC VARIABLES
def boxplot(dataframe: pd.DataFrame) -> Figure:
    numb_df_cols = dataframe.select_dtypes(include="number").columns
    print(numb_df_cols)
    fig, axes = plt.subplots(figsize=(10, 5), ncols=numb_df_cols.size)
    for col_name, ax in zip(numb_df_cols, axes, strict=True):
        dataframe[col_name].plot(
            kind="box", ax=ax, title=col_name.replace("_", " ").title()
        )
        stats = boxplot_stats(dataframe[col_name])[0]
        yticks = [
            stats["whislo"],
            stats["q1"],
            stats["med"],
            stats["q3"],
            stats["whishi"],
        ]
        ax.set_yticks(ticks=yticks, labels=[f"{tick:.2f}" for tick in yticks])
        # GET THE COLUMNS VALUES MEAN
        print(f"Mean ({col_name.replace('_', ' ').title()}): {df[col_name].mean()}")
    fig.tight_layout(w_pad=2.5)
    return fig


fig = boxplot(dataframe=df)
plt.show()
plt.close(fig=fig)

In [ ]:
# PLOT HISTOGRAM FOR NUMERIC VARIABLES
def histograms(dataframe: pd.DataFrame) -> Figure:
    numb_df_cols = dataframe.select_dtypes(include="number").columns
    fig, axes = plt.subplots(figsize=(7, 15), nrows=len(numb_df_cols))
    print(axes)
    for col_name, ax in zip(numb_df_cols, axes):
        series = dataframe[col_name]
        nbins = np.ceil(np.log2(series.size)) + 1
        k = (series.max() - series.min()) / 14
        series.plot.hist(bins=int(nbins), edgecolor="black", ax=ax)
        ax.set_title(f"{col_name.replace('_', ' ').title()}")
        ax.set_xlabel(f"{col_name.replace('_', ' ').title()} Values")
        ax.set_xticks([series.min() + i * k for i in range(int(nbins) + 1)])
        ax.tick_params(labelsize=7)
    fig.tight_layout(h_pad=2.5)
    return fig


fig = histograms(dataframe=df)
plt.show()
plt.close(fig)

In [ ]:
# GET EACH SUB FEATURE FREQUENCY FROM EACH CATEGORICAL FEATURE
def get_categorical_frequencies(dataframe: pd.DataFrame) -> pd.DataFrame:
    str_boolean_cols = dataframe.select_dtypes(include=["string", "boolean"]).columns

    categ_freq = []
    for col in str_boolean_cols:
        freq = dataframe[col].value_counts(sort=True).reset_index()
        freq.columns = ["Sub_feature", "Frequency"]
        freq.insert(0, "Feature", col)
        categ_freq.append(freq)

    concat_categ_freq = pd.concat(objs=categ_freq)
    return concat_categ_freq


get_categorical_frequencies(df)

In [ ]:
# GET DATAFRAME GROUPED BY FEATURE AND CHURN
def grouped_feature_churn(dataframe: pd.DataFrame) -> pd.DataFrame:
    dataframe = deepcopy(dataframe)
    str_bool_cols = dataframe.select_dtypes(include=["string", "boolean"]).columns
    str_bool_cols = str_bool_cols.drop("churn")
    grouped_feat_churn_melted = []
    for col in str_bool_cols:
        df_grouped = dataframe.groupby([col, "churn"]).size().reset_index(name="count")
        df_grouped.columns = ["Sub_feature", "Churn", "Frequency"]
        df_grouped.insert(loc=0, column="Feature", value=col)
        grouped_feat_churn_melted.append(df_grouped)

    return pd.concat(grouped_feat_churn_melted)


grouped_feature_churn(dataframe=df)